# HCLTech Career Shaper Programme - Project No. 32
## Credit Card Fraud Detection Using Anomaly Detection
### Exploratory Data Analysis (EDA) Notebook

**Domain:** Digital-Data-AI / AIML  
**Dataset:** dataset.csv (5,000 transactions, 3.5% fraud rate)  
**Deliverable:** Exploratory Analysis with Visualisations


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline


### 1. Ingest Dataset & Inspect Structure

In [ ]:
df = pd.read_csv('dataset.csv')
df['datetime'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['datetime'].dt.hour
df['log_amount'] = np.log1p(df['amount'])

print(f'Total records: {len(df):,}')
print(f'Missing values: {df.isnull().sum().sum()}')
df.head(5)


### 2. Visualisation 1: Class Imbalance & Transaction Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
palette = ['#2b5c8f', '#d9534f']
sns.countplot(data=df, x='is_fraud', ax=axes[0], palette=palette, hue='is_fraud', legend=False)
axes[0].set_title('Transaction Count by Class Label', fontsize=13, fontweight='bold')
axes[0].set_xlabel('is_fraud (0 = Legitimate, 1 = Fraud)')
axes[0].set_ylabel('Transaction Count')

fraud_counts = df['is_fraud'].value_counts()
axes[1].pie(fraud_counts, labels=['Legitimate (96.5%)', 'Fraud (3.5%)'],
            colors=['#2b5c8f', '#d9534f'], autopct='%1.1f%%', startangle=140,
            explode=(0, 0.1), wedgeprops=dict(width=0.45, edgecolor='w', linewidth=2))
axes[1].set_title('Class Imbalance Proportion', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 3. Visualisation 2: Transaction Amount Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.kdeplot(data=df[df['is_fraud']==0]['log_amount'], ax=axes[0], label='Legitimate', color='#2b5c8f', fill=True, alpha=0.35)
sns.kdeplot(data=df[df['is_fraud']==1]['log_amount'], ax=axes[0], label='Fraud', color='#d9534f', fill=True, alpha=0.45)
axes[0].set_title('Log Amount Density Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('log(1 + Amount)')
axes[0].legend()

p99 = df['amount'].quantile(0.99)
sns.boxplot(data=df[df['amount'] <= p99], x='is_fraud', y='amount', ax=axes[1], palette=palette, hue='is_fraud', legend=False)
axes[1].set_title('Amount Distribution (Up to 99th Percentile)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 4. Visualisation 3: Circadian Temporal Trends (Hourly Pattern)

In [ ]:
hourly_stats = df.groupby('hour').agg(
    total_txns=('is_fraud', 'count'),
    fraud_txns=('is_fraud', 'sum')
).reset_index()
hourly_stats['fraud_rate'] = (hourly_stats['fraud_txns'] / hourly_stats['total_txns']) * 100

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(hourly_stats['hour'], hourly_stats['total_txns'], color='#2b5c8f', alpha=0.6, width=0.7)
ax1.set_xlabel('Hour of Day (0 to 23)')
ax1.set_ylabel('Transaction Volume', color='#2b5c8f')
ax1.set_xticks(range(0, 24))

ax2 = ax1.twinx()
ax2.plot(hourly_stats['hour'], hourly_stats['fraud_rate'], color='#d9534f', marker='o', linewidth=2.5)
ax2.set_ylabel('Fraud Rate (%)', color='#d9534f')
ax2.grid(False)
plt.title('Circadian Patterns: Transaction Volume vs. Fraud Rate by Hour', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 5. Visualisation 4: Merchant Category & Card Network Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

cat_summary = (df.groupby('merchant_category')['is_fraud'].mean() * 100).sort_values()
axes[0].barh(cat_summary.index, cat_summary.values, color='#4a90e2')
axes[0].set_title('Fraud Rate by Merchant Category (%)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Fraud Rate (%)')

card_summary = (df.groupby('card_type')['is_fraud'].mean() * 100).sort_values()
axes[1].barh(card_summary.index, card_summary.values, color='#50e3c2')
axes[1].set_title('Fraud Rate by Card Type (%)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Fraud Rate (%)')
plt.tight_layout()
plt.show()
